# Netflix VOID - Video Object and Interaction Deletion

This notebook performs object removal from videos using Netflix's [VOID](https://github.com/Netflix/void-model) model.

## Pipeline

| Step | Description | Tools |
|------|-------------|-------|
| 1 | Setup and model download | HuggingFace, pip |
| 2 | Upload video + mask + prompt | Prebuilt files or create from scratch |
| 3 | Inference (Pass 1) | CogVideoX + VOID checkpoint |
| 4 | View results | IPython Video |

**Requirements:** 40GB+ VRAM GPU (A100 recommended)

**L40S cold-start note:** Loading the CogVideoX 5B transformer can take about 40-50 seconds before denoising starts. Prepare every sequence first, then run them together so the transformer is loaded once per predictor process.

**References:**
- [Paper](https://arxiv.org/abs/2604.02296) | [GitHub](https://github.com/Netflix/void-model) | [HuggingFace](https://huggingface.co/netflix/void-model)

---
## 1. Setup

In [ ]:
import os, sys, subprocess, time

def run(cmd):
    subprocess.run(cmd, shell=True, check=True)

# GPU check
import torch
assert torch.cuda.is_available(), "GPU not found! Runtime > Change runtime type > GPU"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} | VRAM: {vram:.0f} GB")

# System dependencies
run("apt-get -qq update && apt-get -qq install -y ffmpeg git")
run(f"{sys.executable} -m pip install -q --upgrade pip")
run(f"{sys.executable} -m pip install -q huggingface_hub hf_transfer")

In [ ]:
# Clone repo and install dependencies. Reuse the checkout on reruns.
if os.path.exists("/content/void-model/.git"):
    run("git -C /content/void-model pull --ff-only")
else:
    run("git clone https://github.com/Netflix/void-model.git /content/void-model")

os.chdir("/content/void-model")
run(f"{sys.executable} -m pip install -q -r requirements.txt")

---
## 2. Model Download

In [ ]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download, hf_hub_download

# Base model (CogVideoX)
snapshot_download(
    repo_id="alibaba-pai/CogVideoX-Fun-V1.5-5b-InP",
    local_dir="./CogVideoX-Fun-V1.5-5b-InP",
    local_dir_use_symlinks=False,
    resume_download=True,
)

# VOID Pass 1 checkpoint
hf_hub_download(
    repo_id="netflix/void-model",
    filename="void_pass1.safetensors",
    local_dir=".",
    local_dir_use_symlinks=False,
)

# Validate
assert os.path.exists("./CogVideoX-Fun-V1.5-5b-InP/transformer/config.json"), "Base model is missing!"
assert os.path.exists("./void_pass1.safetensors"), "VOID checkpoint is missing!"

---
## 3. Upload Video, Mask, and Prompt

The VOID model expects 3 files for each video:

```
my_video/
├── input_video.mp4      # Source video
├── quadmask_0.mp4       # 4-value mask (0=remove, 63=overlap, 127=affected, 255=preserve)
└── prompt.json          # {"bg": "Scene description after object removal"}
```

### Option A: Upload existing files
Use this cell if you already have mask and prompt files.

### Option B: Create from scratch
If you want to create a mask for your own video, use the [full pipeline notebook](link).

In [ ]:
from google.colab import files
from pathlib import Path
from IPython.display import Video, display
import shutil, json

# ============================================================
# SETTINGS
# ============================================================
SEQ_NAME = "my_video"  # Change and rerun this cell for each additional video directory.
DATA_ROOT = Path("/content/void-model/custom_data")
DATA_DIR = DATA_ROOT / SEQ_NAME
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Upload files
uploaded = files.upload()

for name, content in uploaded.items():
    dest = DATA_DIR / name
    with open(dest, 'wb') as f:
        f.write(content)

# Validation
required = ["input_video.mp4", "quadmask_0.mp4", "prompt.json"]
all_ok = all((DATA_DIR / f).exists() for f in required)

if all_ok:
    with open(DATA_DIR / "prompt.json") as f:
        prompt_data = json.load(f)
    print(f"Prompt: {prompt_data['bg']}")
    print("Input video:")
    display(Video(str(DATA_DIR / "input_video.mp4"), embed=True, width=672))
    print("Quadmask:")
    display(Video(str(DATA_DIR / "quadmask_0.mp4"), embed=True, width=672))
else:
    raise FileNotFoundError("Missing files! Please upload all required files.")

# Any valid sequence folder under custom_data will be run in one predictor process.
# This amortizes the CogVideoX transformer load across every prepared sequence.
prepared_sequences = sorted(
    p.name for p in DATA_ROOT.iterdir()
    if p.is_dir() and all((p / f).exists() for f in required)
)
RUN_SEQS = ",".join(prepared_sequences)
print(f"Prepared sequences for one-process inference: {RUN_SEQS}")

---
## 4. Inference (Pass 1)

Run all prepared sequences in one predictor process. The upstream script loads the CogVideoX transformer once, then iterates over `run_seqs`, so batching avoids paying the L40S cold-start load for every sequence.

In [ ]:
import subprocess, sys, time
os.chdir("/content/void-model")

assert RUN_SEQS, "No prepared sequences found. Run the upload cell first."

cmd = [
    sys.executable,
    "inference/cogvideox_fun/predict_v2v.py",
    "--config", "config/quadmask_cogvideox.py",
    "--config.data.data_rootdir=/content/void-model/custom_data",
    f"--config.experiment.run_seqs={RUN_SEQS}",
    "--config.experiment.save_path=/content/void_outputs",
    "--config.video_model.transformer_path=./void_pass1.safetensors",
]

print(f"Running sequences in one predictor process: {RUN_SEQS}")
print("The first part of this wall time includes CogVideoX transformer cold-start loading.")
start = time.perf_counter()
subprocess.run(cmd, check=True)
elapsed = time.perf_counter() - start
print(f"VOID predictor wall time: {elapsed:.1f}s")

---
## 5. View Results

In [ ]:
import glob
from IPython.display import Video, display

videos = sorted(glob.glob("/content/void_outputs/**/*.mp4", recursive=True))

for v in videos:
    name = os.path.basename(v)
    if "tuple" in name:
        print("Comparison (input | mask | output):")
        display(Video(v, embed=True, width=1344))
    else:
        print("Output:")
        display(Video(v, embed=True, width=672))

---
## How to Create a Quadmask?

If you want to create a mask from scratch for your own video:

### Method 1: Full VLM Pipeline (Recommended)
Automatically generates a quadmask using SAM2 + Gemini/Groq VLM.

```python
# 1. Object segmentation with SAM2
# 2. Interaction analysis with VLM (Gemini or Groq Llama 4 Scout)
# 3. Quadmask generation (0=remove, 63=overlap, 127=affected, 255=preserve)
```

For details, check the `VLM-MASK-REASONER/` directory.

### Method 2: Manual SAM2 Mask
Creates only a binary mask with SAM2 (without interaction analysis).

```python
from sam2.build_sam import build_sam2_video_predictor
# Select object with points -> propagate to all frames -> quadmask video
```

### Quadmask Values
| Value | Meaning | Example |
|-------|---------|---------|
| 0 (black) | Object to remove | Ice cream truck |
| 63 (dark gray) | Overlap region | Object-ground boundary |
| 127 (gray) | Affected region | Shadow, reflection |
| 255 (white) | Preserved background | Road, pedestrians |